In [1]:
import pandas as pd

In [78]:
import os
import pandas as pd

# ------------------------------
# User-Defined Variables
# ------------------------------

# Retention time window: change this value to adjust the allowed difference
RETENTION_TIME_WINDOW = 0.8

# Input file paths
FAME_CSV_PATH = 'Projects/FAME/analysis/OFF/off_possible/FAME_RT.csv'
NIST_PARQUET_PATH = 'Projects/NIST/isomer_filter/NIST_n3_isomer_filtered.parquet'

# Output directory and file suffixes (you can change the output directory if needed)
OUTPUT_DIR = 'Projects/NIST/isomer_filter'  # or any directory you prefer
KEEP_SUFFIX = '_FAME_KEEP.csv'
DONT_KEEP_SUFFIX = '_FAMEDONTKEEP.csv'

# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------
# 1. Load FAME CSV and NIST Parquet
# ------------------------------

try:
    df_fame = pd.read_csv(FAME_CSV_PATH)
    print(f"[DEBUG] Successfully loaded FAME CSV from: {FAME_CSV_PATH}")
    print(f"[DEBUG] FAME DataFrame shape: {df_fame.shape}")
except Exception as e:
    print(f"[ERROR] Failed to load FAME CSV: {e}")
    raise

try:
    df_nist = pd.read_parquet(NIST_PARQUET_PATH)
    print(f"[DEBUG] Successfully loaded NIST Parquet from: {NIST_PARQUET_PATH}")
    print(f"[DEBUG] NIST DataFrame shape: {df_nist.shape}")
except Exception as e:
    print(f"[ERROR] Failed to load NIST Parquet: {e}")
    raise

# ------------------------------
# 1a. (Optional) Handling duplicates in FAME:
# ------------------------------
# In the previous code we dropped duplicates by Species.
# However, if a species appears in two isomeric forms (e.g., cis and trans),
# we want to keep both rows (each with its own retention time window).
# So we will no longer drop duplicates from FAME.
# If you wish to drop duplicates in some other way, modify here.
#
# For example, you might choose to keep both rows if the FAME file distinguishes
# between isomers via an "Isomer" column.
#
# ------------------------------
# 2. Prepare FAME DataFrame for merging
# ------------------------------
# To avoid confusion between the NIST and FAME retention times, rename the FAME retention time column.
df_fame_merge = df_fame[['Species', 'Retention_Time', 'Isomer']].rename(
    columns={'Retention_Time': 'FAME_Retention_Time'}
)

# ------------------------------
# 3. Merge NIST with FAME on Species (and allow for multiple matches per species)
# ------------------------------
# Since the FAME file can have more than one row per species (e.g., cis and trans),
# we want to allow the merge to return multiple rows per NIST row.
# To later recover unique NIST rows, we add an identifier (orig_index) to the NIST DataFrame.
df_nist_reset = df_nist.reset_index().rename(columns={'index': 'orig_index'})

try:
    merged = df_nist_reset.merge(
        df_fame_merge,
        on='Species',
        how='left'
    )
    print(f"[DEBUG] After merge, merged DataFrame shape: {merged.shape}")
except Exception as e:
    print(f"[ERROR] Merge failed: {e}")
    raise

# ------------------------------
# 4. Check Retention Time for Each NIST Row Across FAME Matches
# ------------------------------
# For a given NIST row (identified by orig_index), there may be multiple FAME matches.
# We will mark the NIST row as "keep" if any of the following is true:
#   - No matching FAME retention time was found (i.e. FAME_Retention_Time is NaN)
#   - The absolute difference between the NIST retention time (Retention_Time)
#     and the FAME retention time (FAME_Retention_Time) is <= RETENTION_TIME_WINDOW.
#
# We first create a temporary column 'match_ok' that is True for an individual merge row if:
#   (pd.isna(FAME_Retention_Time)) or (abs(NIST_RT - FAME_Retention_Time) <= window)
merged['match_ok'] = merged.apply(
    lambda row: True if pd.isna(row['FAME_Retention_Time']) 
                else abs(row['Retention_Time'] - row['FAME_Retention_Time']) <= RETENTION_TIME_WINDOW,
    axis=1
)

# Now, group by the original NIST row and mark that NIST row as keep if any of the merged rows had a match.
keep_mapping = merged.groupby('orig_index')['match_ok'].any()

# ------------------------------
# 5. Split the NIST DataFrame Based on the Check
# ------------------------------
# Now we join the keep mapping back to the original NIST DataFrame.
df_nist_reset['keep'] = df_nist_reset['orig_index'].map(keep_mapping)

# Create two DataFrames:
#   df_FAME_KEEP: NIST rows that either did not have a FAME match (i.e. no FAME retention time available)
#                  OR had at least one FAME match with a retention time within the allowed window.
#   df_FAMEDONTKEEP: NIST rows that had FAME matches but none within the allowed window.
df_FAME_KEEP = df_nist_reset[df_nist_reset['keep']].copy()
df_FAMEDONTKEEP = df_nist_reset[~df_nist_reset['keep']].copy()

# Optionally, if you wish to drop the added columns from the output, do so:
for df in [df_FAME_KEEP, df_FAMEDONTKEEP]:
    for col in ['orig_index', 'keep']:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)

print(f"[DEBUG] df_FAME_KEEP shape: {df_FAME_KEEP.shape}")
print(f"[DEBUG] df_FAMEDONTKEEP shape: {df_FAMEDONTKEEP.shape}")

# ------------------------------
# 6. Save the Two DataFrames to CSV Files
# ------------------------------
# Generate output filenames based on the original NIST file's base name.
base_filename = os.path.splitext(os.path.basename(NIST_PARQUET_PATH))[0]
keep_filepath = os.path.join(OUTPUT_DIR, base_filename + KEEP_SUFFIX)
dontkeep_filepath = os.path.join(OUTPUT_DIR, base_filename + DONT_KEEP_SUFFIX)

try:
    df_FAME_KEEP.to_csv(keep_filepath, index=False)
    print(f"[DEBUG] Saved df_FAME_KEEP to: {keep_filepath}")
except Exception as e:
    print(f"[ERROR] Failed to save df_FAME_KEEP: {e}")

try:
    df_FAMEDONTKEEP.to_csv(dontkeep_filepath, index=False)
    print(f"[DEBUG] Saved df_FAMEDONTKEEP to: {dontkeep_filepath}")
except Exception as e:
    print(f"[ERROR] Failed to save df_FAMEDONTKEEP: {e}")

# ------------------------------
# Additional Note
# ------------------------------
# If you still see the same species (lipid) appearing in both output DataFrames,
# consider that the NIST dataset might have multiple rows for the same species with varying retention times.
# This approach uses a group-by on the original NIST row so that if any FAME window (e.g., cis or trans)
# qualifies, the entire NIST entry is kept.


[DEBUG] Successfully loaded FAME CSV from: Projects/FAME/analysis/OFF/off_possible/FAME_RT.csv
[DEBUG] FAME DataFrame shape: (10, 6)
[DEBUG] Successfully loaded NIST Parquet from: Projects/NIST/isomer_filter/NIST_n3_isomer_filtered.parquet
[DEBUG] NIST DataFrame shape: (4575, 38)
[DEBUG] After merge, merged DataFrame shape: (4803, 41)
[DEBUG] df_FAME_KEEP shape: (4489, 38)
[DEBUG] df_FAMEDONTKEEP shape: (86, 38)
[DEBUG] Saved df_FAME_KEEP to: Projects/NIST/isomer_filter/NIST_n3_isomer_filtered_FAME_KEEP.csv
[DEBUG] Saved df_FAMEDONTKEEP to: Projects/NIST/isomer_filter/NIST_n3_isomer_filtered_FAMEDONTKEEP.csv


In [80]:
df_keep = pd.read_csv('Projects/NIST/isomer_filter/NIST_n3_isomer_filtered_FAME_KEEP.csv')
df_dontkeep = pd.read_csv('Projects/NIST/isomer_filter/NIST_n1_isomer_filtered_FAMEDONTKEEP.csv')

df_keep

# look at 18:1 Species column
df_keep[df_keep['Species'].str.contains('18:1')]

,Lipid,Retention_Time,OzESI_Intensity,group_by_ion,group_by_lipid,Sample_ID,Transition,Sample,Parent_Ion,Product_Ion,...,Baseline_Drift,Shoulder_Count,Biology,Genotype,Cage,Mouse,Normalized_Peak_Area,n_value,n_position,OzOFF_Isomer
1295,FA(18:1)_<>_n-2,12.405583,47830.0,74,317,02032025_n3Plasma_AMP_ozoneonv2,437.4 -> 183.0,NIST_n3,437.4,183.0,...,-20980.105201,1,NaN,NaN,NaN,NaN,0.164084,2,2.0,trans
1296,FA(18:1)_<>_n-4,12.359633,7458.0,62,319,02032025_n3Plasma_AMP_ozoneonv2,409.3 -> 183.0,NIST_n3,409.3,183.0,...,6189.991835,1,NaN,NaN,NaN,NaN,0.020574,4,4.0,trans
1297,FA(18:1)_<>_n-4,12.581550,4490.0,62,319,02032025_n3Plasma_AMP_ozoneonv2,409.3 -> 183.0,NIST_n3,409.3,183.0,...,15236.593060,0,NaN,NaN,NaN,NaN,0.002375,4,4.0,trans
1298,FA(18:1)_<>_n-5,12.296950,8869.0,56,320,02032025_n3Plasma_AMP_ozoneonv2,395.3 -> 183.0,NIST_n3,395.3,183.0,...,-563.782511,2,NaN,NaN,NaN,NaN,0.036786,5,5.0,trans
1299,FA(18:1)_<>_n-5,12.407917,8774.0,56,320,02032025_n3Plasma_AMP_ozoneonv2,395.3 -> 183.0,NIST_n3,395.3,183.0,...,18730.725020,0,NaN,NaN,NaN,NaN,0.007120,5,5.0,trans
1300,FA(18:1)_<>_n-6,12.313533,25533.0,50,321,02032025_n3Plasma_AMP_ozoneonv2,381.3 -> 183.0,NIST_n3,381.3,183.0,...,3836.104306,2,NaN,NaN,NaN,NaN,0.091578,6,6.0,trans
1301,FA(18:1)_<>_n-6,12.456200,23759.0,50,321,02032025_n3Plasma_AMP_ozoneonv2,381.3 -> 183.0,NIST_n3,381.3,183.0,...,79091.216936,0,NaN,NaN,NaN,NaN,0.019130,6,6.0,trans
1302,FA(18:1)_<>_n-6,12.519600,16750.0,50,321,02032025_n3Plasma_AMP_ozoneonv2,381.3 -> 183.0,NIST_n3,381.3,183.0,...,17160.883281,0,NaN,NaN,NaN,NaN,0.009290,6,6.0,trans
1303,FA(18:1)_<>_n-7,12.155733,20326.0,44,322,02032025_n3Plasma_AMP_ozoneonv2,367.3 -> 183.0,NIST_n3,367.3,183.0,...,19996.913080,0,NaN,NaN,NaN,NaN,0.016809,7,7.0,trans
1304,FA(18:1)_<>_n-7,12.361817,49197.0,44,322,02032025_n3Plasma_AMP_ozoneonv2,367.3 -> 183.0,NIST_n3,367.3,183.0,...,12850.729052,1,NaN,NaN,NaN,NaN,0.167056,7,7.0,trans


In [71]:
# print unique species in df_keep
df_keep['OzOFF_Isomer'].unique()

array(['trans', 'cis', nan], dtype=object)

# Function take the highest values

In [76]:
# import pandas as pd

# def filter_highest_intensity(df):
#     """
#     For each unique combination of 'Lipid', 'n_value', and 'OzOFF_Isomer' in the DataFrame,
#     return the row with the highest 'OzESI_Intensity'.
    
#     Parameters:
#         df (pd.DataFrame): DataFrame containing the columns 'Lipid', 'n_value', 'OzOFF_Isomer', and 'OzESI_Intensity'.
        
#     Returns:
#         pd.DataFrame: A filtered DataFrame with only the highest intensity row for each group.
#     """
#     # Check that required columns exist in the DataFrame
#     required_cols = {'Lipid', 'n_value', 'OzOFF_Isomer', 'OzESI_Intensity'}
#     if not required_cols.issubset(df.columns):
#         raise ValueError(f"DataFrame must contain columns: {required_cols}")
    
#     # Group by the combination of columns and select the index of the max OzESI_Intensity in each group.
#     idx = df.groupby(['Lipid', 'n_value', 'OzOFF_Isomer'])['OzESI_Intensity'].idxmax()
#     # Return the corresponding rows
#     return df.loc[idx].reset_index(drop=True)


# # Example usage:
# if __name__ == "__main__":
#     # Read in the CSV files
#     df_keep = pd.read_csv('Projects/NIST/isomer_filter/NIST_n1_isomer_filtered_FAME_KEEP.csv')
#     df_dontkeep = pd.read_csv('Projects/NIST/isomer_filter/NIST_n1_isomer_filtered_FAMEDONTKEEP.csv')
    
#     # Replace NaN values in 'OzOFF_Isomer' with "unknown"
#     df_keep["OzOFF_Isomer"] = df_keep["OzOFF_Isomer"].fillna("unknown")
#     df_dontkeep["OzOFF_Isomer"] = df_dontkeep["OzOFF_Isomer"].fillna("unknown")
    
#     # Optionally filter for species containing '18:1' if needed
#     # df_keep_18_1 = df_keep[df_keep['Species'].str.contains('18:1', na=False)]
    
#     # Filter for the highest OzESI_Intensity rows based on the grouping
#     df_keep_filtered = filter_highest_intensity(df_keep)
    
#     # Display the filtered DataFrame
#     print(df_keep_filtered)


In [81]:
import os
import glob
import pandas as pd

def filter_highest_intensity(df):
    """
    For each unique combination of 'Lipid', 'n_value', and 'OzOFF_Isomer' in the DataFrame,
    return the row with the highest 'OzESI_Intensity'.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing the columns 'Lipid', 'n_value', 'OzOFF_Isomer', and 'OzESI_Intensity'.
        
    Returns:
        pd.DataFrame: A filtered DataFrame with only the highest intensity row for each group.
    """
    # Check that required columns exist in the DataFrame
    required_cols = {'Lipid', 'n_value', 'OzOFF_Isomer', 'OzESI_Intensity'}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"DataFrame must contain columns: {required_cols}")
    
    # Group by the combination of columns and select the index of the max OzESI_Intensity in each group.
    idx = df.groupby(['Lipid', 'n_value', 'OzOFF_Isomer'])['OzESI_Intensity'].idxmax()
    
    # Return the corresponding rows
    return df.loc[idx].reset_index(drop=True)

if __name__ == "__main__":
    # Define the base directory and the output subdirectory
    base_dir = "Projects/NIST/isomer_filter"
    output_dir = os.path.join(base_dir, "top2")
    os.makedirs(output_dir, exist_ok=True)
    
    # Pattern to match all CSV files that contain "FAME_KEEP" in the filename
    file_pattern = os.path.join(base_dir, "*FAME_KEEP*.csv")
    csv_files = glob.glob(file_pattern)
    
    if not csv_files:
        print(f"No files matching pattern {file_pattern} were found.")
    
    # Process each CSV file
    for file_path in csv_files:
        try:
            # Read the CSV file into a DataFrame
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            continue

        # Replace NaN values in the 'OzOFF_Isomer' column with "unknown"
        if 'OzOFF_Isomer' in df.columns:
            df['OzOFF_Isomer'] = df['OzOFF_Isomer'].fillna("unknown")
        else:
            print(f"Warning: 'OzOFF_Isomer' column not found in {file_path}.")
        
        try:
            # Filter the DataFrame to get rows with the highest OzESI_Intensity per group
            df_filtered = filter_highest_intensity(df)
        except Exception as e:
            print(f"Error filtering {file_path}: {e}")
            continue
        
        # Define the output file path
        filename = os.path.basename(file_path)
        output_file = os.path.join(output_dir, filename)
        
        # Save the filtered DataFrame to the output CSV file
        try:
            df_filtered.to_csv(output_file, index=False)
            print(f"Saved filtered file to {output_file}")
        except Exception as e:
            print(f"Error saving {output_file}: {e}")


Saved filtered file to Projects/NIST/isomer_filter/top2/NIST_n2_isomer_filtered_FAME_KEEP.csv
Saved filtered file to Projects/NIST/isomer_filter/top2/NIST_n3_isomer_filtered_FAME_KEEP.csv
Saved filtered file to Projects/NIST/isomer_filter/top2/NIST_n1_isomer_filtered_FAME_KEEP.csv


In [99]:
top2 = pd.read_csv('Projects/NIST/isomer_filter/top2/NIST_n1_isomer_filtered_FAME_KEEP.csv')
#print unique Species in top2
print(top2['Species'].unique())
top2

['10:1' '10:2' '11:1' '11:2' '12:1' '12:2' '13:1' '13:2' '14:1' '14:2'
 '15:1' '16:1' '16:2' '17:1' '17:2' '17:5' '18:1' '18:2' '18:3' '18:4'
 '19:1' '19:2' '19:3' '20:1' '20:2' '20:3' '20:4' '20:5' '21:1' '22:1'
 '22:2' '22:3' '22:4' '22:5' '22:6' '23:2' '6:1' '7:1' '8:1' '9:1']


,Lipid,Retention_Time,OzESI_Intensity,group_by_ion,group_by_lipid,Sample_ID,Transition,Sample,Parent_Ion,Product_Ion,...,Baseline_Drift,Shoulder_Count,Biology,Genotype,Cage,Mouse,Normalized_Peak_Area,n_value,n_position,OzOFF_Isomer
0,FA(10:1)_<>_n-2,9.700933,4438.0,26,0,02032025_n1Plasma_AMP_ozoneonv2,325.2 -> 183.0,NIST_n1,325.2,183.0,...,2590.431892,2,NaN,NaN,NaN,NaN,0.016238,2,2.0,cis
1,FA(10:1)_<>_n-2,11.666517,6340.0,26,0,02032025_n1Plasma_AMP_ozoneonv2,325.2 -> 183.0,NIST_n1,325.2,183.0,...,29806.369382,0,NaN,NaN,NaN,NaN,0.004753,2,2.0,trans
2,FA(10:1)_<>_n-3,9.669950,3914.0,20,1,02032025_n1Plasma_AMP_ozoneonv2,311.3 -> 183.0,NIST_n1,311.3,183.0,...,15053.627760,0,NaN,NaN,NaN,NaN,0.004306,3,3.0,cis
3,FA(10:1)_<>_n-3,11.857433,1106.0,20,1,02032025_n1Plasma_AMP_ozoneonv2,311.3 -> 183.0,NIST_n1,311.3,183.0,...,5114.353312,1,NaN,NaN,NaN,NaN,0.001163,3,3.0,trans
4,FA(10:1)_<>_n-4,9.686517,8315.0,14,2,02032025_n1Plasma_AMP_ozoneonv2,297.2 -> 183.0,NIST_n1,297.2,183.0,...,8416.403785,0,NaN,NaN,NaN,NaN,0.009829,4,4.0,cis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
852,FA(9:1)_<>_n-3,11.778900,562.0,14,1146,02032025_n1Plasma_AMP_ozoneonv2,297.2 -> 183.0,NIST_n1,297.2,183.0,...,554.653763,1,NaN,NaN,NaN,NaN,0.000756,3,3.0,trans
853,FA(9:1)_<>_n-4,9.956583,29154.0,9,1147,02032025_n1Plasma_AMP_ozoneonv2,283.2 -> 183.0,NIST_n1,283.2,183.0,...,-17858.765608,1,NaN,NaN,NaN,NaN,0.235224,4,4.0,cis
854,FA(9:1)_<>_n-4,11.731933,2054.0,9,1147,02032025_n1Plasma_AMP_ozoneonv2,283.2 -> 183.0,NIST_n1,283.2,183.0,...,5198.738170,2,NaN,NaN,NaN,NaN,0.003578,4,4.0,trans
855,FA(9:1)_<>_n-5,9.750983,15459.0,5,1148,02032025_n1Plasma_AMP_ozoneonv2,269.1 -> 183.0,NIST_n1,269.1,183.0,...,54880.576836,1,NaN,NaN,NaN,NaN,0.024640,5,5.0,cis


In [105]:
# filter top2 based on 18:1 Species
top2[top2['Species'].str.contains('22:6')]

,Lipid,Retention_Time,OzESI_Intensity,group_by_ion,group_by_lipid,Sample_ID,Transition,Sample,Parent_Ion,Product_Ion,...,Baseline_Drift,Shoulder_Count,Biology,Genotype,Cage,Mouse,Normalized_Peak_Area,n_value,n_position,OzOFF_Isomer
738,FA(22:6)_<BBBBB>_n-10,10.871667,1494.0,45,994,02032025_n1Plasma_AMP_ozoneonv2,371.3 -> 183.0,NIST_n1,371.3,183.0,...,9712.420231,0,NaN,NaN,NaN,NaN,0.002322,10,10.0,unknown
739,FA(22:6)_<BBBBB>_n-11,9.921300,36867.0,39,995,02032025_n1Plasma_AMP_ozoneonv2,357.2 -> 183.0,NIST_n1,357.2,183.0,...,-22342.517787,1,NaN,NaN,NaN,NaN,0.197687,11,11.0,unknown
740,FA(22:6)_<BBBBB>_n-12,9.747650,20796.0,33,996,02032025_n1Plasma_AMP_ozoneonv2,343.2 -> 183.0,NIST_n1,343.2,183.0,...,64677.744590,1,NaN,NaN,NaN,NaN,0.032460,12,12.0,unknown
741,FA(22:6)_<BBBBB>_n-13,9.827633,528.0,27,997,02032025_n1Plasma_AMP_ozoneonv2,329.2 -> 183.0,NIST_n1,329.2,183.0,...,5962.145110,0,NaN,NaN,NaN,NaN,0.000526,13,13.0,unknown
742,FA(22:6)_<BBBBB>_n-2,9.692667,632.0,93,1000,02032025_n1Plasma_AMP_ozoneonv2,483.3 -> 183.0,NIST_n1,483.3,183.0,...,5526.813880,0,NaN,NaN,NaN,NaN,0.000722,2,2.0,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
805,FA(22:6)_<FFFFF>_n-18,9.750983,15459.0,5,1076,02032025_n1Plasma_AMP_ozoneonv2,269.1 -> 183.0,NIST_n1,269.1,183.0,...,54880.576836,1,NaN,NaN,NaN,NaN,0.024640,18,18.0,unknown
806,FA(22:6)_<FFFFF>_n-6,9.932783,25803.0,74,1078,02032025_n1Plasma_AMP_ozoneonv2,437.4 -> 183.0,NIST_n1,437.4,183.0,...,-9149.499267,1,NaN,NaN,NaN,NaN,0.135171,6,6.0,unknown
807,FA(22:6)_<FFFFF>_n-7,11.011533,4905227.0,68,1079,02032025_n1Plasma_AMP_ozoneonv2,423.3 -> 183.0,NIST_n1,423.3,183.0,...,454497.295151,1,NaN,NaN,NaN,NaN,20.031177,7,7.0,unknown
808,FA(22:6)_<FFFFF>_n-8,9.728300,70059.0,62,1080,02032025_n1Plasma_AMP_ozoneonv2,409.3 -> 183.0,NIST_n1,409.3,183.0,...,85947.232578,1,NaN,NaN,NaN,NaN,0.224696,8,8.0,unknown


In [87]:
# pRINT all unique values of lipid
top2['Species'].unique()

array(['10:1', '10:2', '11:1', '11:2', '12:1', '12:2', '13:1', '13:2',
       '14:1', '14:2', '15:1', '16:1', '16:2', '17:1', '17:2', '17:5',
       '18:1', '18:2', '18:3', '18:4', '19:1', '19:2', '19:3', '20:1',
       '20:2', '20:3', '20:4', '20:5', '21:1', '22:1', '22:2', '22:3',
       '22:4', '22:5', '22:6', '23:2', '6:1', '7:1', '8:1', '9:1'],
      dtype=object)

# BAR PLOT JSUT FAME NO NP

In [88]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------
# User-Defined Variables
# ------------------------------

# Directory where the FAME_KEEP CSV files are stored
INPUT_DIR = 'Projects/NIST/isomer_filter/top2/'

# ------------------------------
# Process Each FAME_KEEP CSV File
# ------------------------------

# List through all files in the input directory that end with '_FAME_KEEP.csv'
for filename in os.listdir(INPUT_DIR):
    if filename.endswith('_FAME_KEEP.csv'):
        file_path = os.path.join(INPUT_DIR, filename)
        print(f"[INFO] Processing file: {file_path}")
        
        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"[ERROR] Could not read {file_path}: {e}")
            continue

        # Create a new subdirectory for the plots.
        # Example: if filename is "unmatched_NIST_n2_FAME_KEEP.csv", the subdir will be "plot_unmatched_NIST_n2_FAME_KEEP"
        base_filename = os.path.splitext(filename)[0]
        plot_subdir = os.path.join(INPUT_DIR, f"plot_{base_filename}")
        os.makedirs(plot_subdir, exist_ok=True)
        print(f"[INFO] Created/Found plot directory: {plot_subdir}")
        
        # Check that necessary columns exist
        required_columns = ['Species', 'Lipid', 'OzESI_Intensity', 'OzOFF_Isomer']
        if not all(col in df.columns for col in required_columns):
            print(f"[WARNING] The file {filename} does not contain the required columns {required_columns}. Skipping.")
            continue
        
        # Get unique species from the dataframe
        unique_species = df['Species'].unique()
        print(f"[DEBUG] Found {len(unique_species)} unique species in {filename}.")
        
        # For each unique species, create a bar plot
        for species in unique_species:
            # Filter the dataframe for the current species
            df_species = df[df['Species'] == species].copy()

            # Check if the species string ends with ":1"
            if str(species).endswith(':1'):
                # Combine Lipid and OzOFF_Isomer for the x-axis label
                df_species['Combo'] = df_species['Lipid'].astype(str) + '_' + df_species['OzOFF_Isomer'].astype(str)
                x_values = df_species['Combo']
                x_label = 'Lipid_OzOFF_Isomer'
            else:
                # Use only the Lipid value for the x-axis label
                x_values = df_species['Lipid']
                x_label = 'Lipid'
            
            # Create the bar plot:
            #   x-axis: determined above
            #   y-axis: OzESI_Intensity values
            plt.figure(figsize=(10, 6))
            plt.bar(x_values, df_species['OzESI_Intensity'], color='skyblue')
            plt.xlabel(x_label, fontsize=12)
            plt.ylabel('OzESI_Intensity', fontsize=12)
            plt.title(f"Species: {species}", fontsize=14)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()

            # Sanitize the species string for use in a filename (replace problematic characters)
            safe_species = str(species).replace(":", "_").replace(" ", "_").replace("/", "_")
            plot_filename = f"{safe_species}.png"
            plot_filepath = os.path.join(plot_subdir, plot_filename)
            
            try:
                plt.savefig(plot_filepath)
                print(f"[INFO] Saved plot for species '{species}' to {plot_filepath}")
            except Exception as e:
                print(f"[ERROR] Could not save plot for species '{species}': {e}")
            finally:
                plt.close()  # Close the figure to free up memory


[INFO] Processing file: Projects/NIST/isomer_filter/top2/NIST_n2_isomer_filtered_FAME_KEEP.csv
[INFO] Created/Found plot directory: Projects/NIST/isomer_filter/top2/plot_NIST_n2_isomer_filtered_FAME_KEEP
[DEBUG] Found 33 unique species in NIST_n2_isomer_filtered_FAME_KEEP.csv.
[INFO] Saved plot for species '10:1' to Projects/NIST/isomer_filter/top2/plot_NIST_n2_isomer_filtered_FAME_KEEP/10_1.png
[INFO] Saved plot for species '11:1' to Projects/NIST/isomer_filter/top2/plot_NIST_n2_isomer_filtered_FAME_KEEP/11_1.png
[INFO] Saved plot for species '12:1' to Projects/NIST/isomer_filter/top2/plot_NIST_n2_isomer_filtered_FAME_KEEP/12_1.png
[INFO] Saved plot for species '12:2' to Projects/NIST/isomer_filter/top2/plot_NIST_n2_isomer_filtered_FAME_KEEP/12_2.png
[INFO] Saved plot for species '13:1' to Projects/NIST/isomer_filter/top2/plot_NIST_n2_isomer_filtered_FAME_KEEP/13_1.png
[INFO] Saved plot for species '13:2' to Projects/NIST/isomer_filter/top2/plot_NIST_n2_isomer_filtered_FAME_KEEP/13_2.

# COMBINE DOUBLE BOND VALUES

In [97]:
import pandas as pd
import numpy as np
import itertools

def n_value_combine_updated(df, retention_time_threshold=0.2):
    """
    Combines lipid n-values based on specified rules:
    - Combines n-values with the same Retention_Time within the threshold.
    - Groups n-values to meet the expected positions of FA(x:y).
    - Adds a 'Copy' column for each combination if necessary.
    - Includes 'Parent_Ion' and 'group_by_lipid' columns in the output, aggregating all relevant values separated by '| '.
    - Rounds Retention_Time to 1 decimal place in the output.

    Parameters:
    - df (DataFrame): Input DataFrame containing 'Lipid', 'Parent_Ion', 'group_by_lipid', and 'Retention_Time' columns.
    - retention_time_threshold (float): Maximum difference in retention time to consider lipids similar.

    Returns:
    - DataFrame: New DataFrame with combined lipid values, rounded Retention Times, 'Copy' column (if applicable),
                 'Parent_Ion' and 'group_by_lipid' columns with values separated by '| ', and 'total_n_values' column indicating the count of n_values per species.
    """
    required_columns = ['Parent_Ion', 'group_by_lipid', 'Lipid', 'Retention_Time']
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Input DataFrame is missing required columns: {', '.join(missing_columns)}")

    df['Parent_Ion'] = df['Parent_Ion'].astype(str)
    df['group_by_lipid'] = df['group_by_lipid'].astype(str)

    lipid_pattern = r'(FA\(\d+:\d+\))_<(.*?)>_n-(\d+)'
    extracted = df['Lipid'].str.extract(lipid_pattern, expand=True)
    if extracted.isnull().values.any():
        raise ValueError("Some entries in 'Lipid' column do not match the expected pattern 'FA(x:y)_<positions>_n-z'.")
    df[['FA', 'positions', 'n_value']] = extracted
    df['n_value'] = df['n_value'].astype(int)

    df['Retention_Time'] = pd.to_numeric(df['Retention_Time'], errors='coerce')
    if df['Retention_Time'].isnull().any():
        raise ValueError("Some 'Retention_Time' values could not be converted to float.")
    df['Retention_Time'] = df['Retention_Time'].round(1)

    total_positions_pattern = r'FA\(\d+:(\d+)\)'
    extracted_positions = df['FA'].str.extract(total_positions_pattern)
    if extracted_positions.isnull().values.any():
        raise ValueError("Some entries in 'FA' column do not match the expected pattern 'FA(x:y)'.")
    df['total_positions'] = extracted_positions[0].astype(int)

    df = df.sort_values(by=['FA', 'Retention_Time', 'n_value']).reset_index(drop=True)

    combined_results = []
    separator = '| '

    for fa, fa_group in df.groupby('FA'):
        expected_positions = fa_group['total_positions'].iloc[0]

        # Cluster by Retention_Time within threshold
        fa_group = fa_group.sort_values(by='Retention_Time').reset_index(drop=True)
        while not fa_group.empty:
            seed = fa_group.iloc[0]
            seed_retention_time = seed['Retention_Time']

            mask = abs(fa_group['Retention_Time'] - seed_retention_time) <= retention_time_threshold
            cluster = fa_group[mask]
            fa_group = fa_group[~mask]

            n_values = sorted(set(cluster['n_value'].tolist()))
            n_values_times = dict(zip(cluster['n_value'], cluster['Retention_Time']))
            n_values_parent_ions = dict(zip(cluster['n_value'], cluster['Parent_Ion']))
            n_values_group_by_lipid = dict(zip(cluster['n_value'], cluster['group_by_lipid']))

            num_n_values = len(n_values)

            if num_n_values > expected_positions:
                combinations_list = list(itertools.combinations(n_values, expected_positions))
                for idx, comb in enumerate(combinations_list, 1):
                    n_values_str = ' '.join([f"n-{n}" for n in comb])
                    retention_times = [str(n_values_times[n]) for n in comb]
                    retention_times_str = ','.join(retention_times)
                    parent_ions = [n_values_parent_ions[n] for n in comb]
                    parent_ions_str = separator.join(parent_ions)
                    group_by_lipids = [n_values_group_by_lipid[n] for n in comb]
                    group_by_lipids_str = separator.join(group_by_lipids)

                    combined_lipid = f"{fa} {n_values_str}"

                    combined_results.append({
                        'Lipid': combined_lipid,
                        'Parent_Ion': parent_ions_str,
                        'group_by_lipid': group_by_lipids_str,
                        'Retention_Time': retention_times_str,
                        'Copy': f"Copy{idx}",
                        'total_n_values': num_n_values
                    })
            else:
                n_values_str = ' '.join([f"n-{n}" for n in n_values])
                combined_lipid = f"{fa} {n_values_str}".strip()
                retention_times = [str(n_values_times[n]) for n in n_values]
                retention_times_str = ','.join(retention_times)
                parent_ions = [n_values_parent_ions[n] for n in n_values]
                parent_ions_str = separator.join(parent_ions)
                group_by_lipids = [n_values_group_by_lipid[n] for n in n_values]
                group_by_lipids_str = separator.join(group_by_lipids)

                combined_results.append({
                    'Lipid': combined_lipid,
                    'Parent_Ion': parent_ions_str,
                    'group_by_lipid': group_by_lipids_str,
                    'Retention_Time': retention_times_str,
                    'Copy': np.nan,
                    'total_n_values': num_n_values
                })

    df_combined_n_values = pd.DataFrame(combined_results)
    df_combined_n_values = df_combined_n_values[['Lipid', 'Parent_Ion', 'group_by_lipid', 'Retention_Time', 'Copy', 'total_n_values']]
    df_combined_n_values['Copy'] = df_combined_n_values['Copy'].replace('', np.nan)

    return df_combined_n_values

d1 = pd.read_csv('Projects/NIST/isomer_filter/top2/NIST_n3_isomer_filtered_FAME_KEEP.csv')
# save in sub dir grouped_doublebond
os.makedirs('Projects/NIST/isomer_filter/top2/grouped_doublebond', exist_ok=True)

#print saving to dir
print('Projects/NIST/isomer_filter/top2/grouped_doublebond/NIST_n3_isomer_filtered_FAME_KEEP.csv')
# Apply the function
df_combined_n_values = n_value_combine_updated(d1)
df_combined_n_values.to_csv('Projects/NIST/isomer_filter/top2/grouped_doublebond/NIST_n1_isomer_filtered_FAME_KEEP.csv', index=False)
# Display the first 60 rows
df_combined_n_values.tail(60)


Projects/NIST/isomer_filter/top2/grouped_doublebond/NIST_n3_isomer_filtered_FAME_KEEP.csv


,Lipid,Parent_Ion,group_by_lipid,Retention_Time,Copy,total_n_values
21713,FA(23:2) n-10 n-15,395.3| 325.2,1192| 1197,"13.1,13.1",Copy68,14
21714,FA(23:2) n-10 n-16,395.3| 311.3,1192| 1198,"13.1,13.0",Copy69,14
21715,FA(23:2) n-10 n-17,395.3| 295.2,1192| 1180,"13.1,13.0",Copy70,14
21716,FA(23:2) n-11 n-12,381.3| 367.3,1193| 1194,"13.0,12.9",Copy71,14
21717,FA(23:2) n-11 n-13,381.3| 353.3,1193| 1195,"13.0,13.1",Copy72,14
21718,FA(23:2) n-11 n-14,381.3| 339.3,1193| 1196,"13.0,13.1",Copy73,14
21719,FA(23:2) n-11 n-15,381.3| 325.2,1193| 1197,"13.0,13.1",Copy74,14
21720,FA(23:2) n-11 n-16,381.3| 311.3,1193| 1198,"13.0,13.0",Copy75,14
21721,FA(23:2) n-11 n-17,381.3| 295.2,1193| 1180,"13.0,13.0",Copy76,14
21722,FA(23:2) n-12 n-13,367.3| 353.3,1194| 1195,"12.9,13.1",Copy77,14


# FAME FILTER AFTER NOT POSSIBLE FILTER

In [29]:
import os
import pandas as pd

# ------------------------------
# User-Defined Variables
# ------------------------------

# Retention time window: change this value to adjust the allowed difference
RETENTION_TIME_WINDOW = 0.5

# FAME CSV file path
FAME_CSV_PATH = 'Projects/FAME/analysis/OFF/off_possible/FAME_RT.csv'

# Input directory for NIST CSV files (all files with "unmatched" in the filename)
INPUT_DIR = 'Projects/NIST/not_possible/'

# Output directory and file suffixes (you can change the output directory if needed)
OUTPUT_DIR = 'Projects/NIST/isomer_filter/AFTER_NP/'  # or any directory you prefer
KEEP_SUFFIX = '_FAME_KEEP.csv'
DONT_KEEP_SUFFIX = '_FAMEDONTKEEP.csv'

# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------
# Load FAME CSV and Prepare Data
# ------------------------------

try:
    df_fame = pd.read_csv(FAME_CSV_PATH)
    print(f"[DEBUG] Successfully loaded FAME CSV from: {FAME_CSV_PATH}")
    print(f"[DEBUG] FAME DataFrame shape (before deduplication): {df_fame.shape}")
except Exception as e:
    print(f"[ERROR] Failed to load FAME CSV: {e}")
    raise

# Remove duplicate species (if any) from FAME
df_fame_unique = df_fame.drop_duplicates(subset='Species')
print(f"[DEBUG] FAME DataFrame shape (after deduplication): {df_fame_unique.shape}")

# ------------------------------
# Define the Retention Time Check Function
# ------------------------------

def check_retention_time(row):
    """
    Returns True if:
      - The FAME retention time is not available (i.e. species not found in FAME), or
      - The absolute difference between the NIST and FAME retention times is within the allowed window.
    Otherwise, returns False.
    """
    if pd.isna(row['Retention_Time_FAME']):
        print(f"[DEBUG] Species '{row['Species']}' not found in FAME. Keeping row.")
        return True
    else:
        rt_diff = abs(row['Retention_Time'] - row['Retention_Time_FAME'])
        if rt_diff <= RETENTION_TIME_WINDOW:
            print(f"[DEBUG] Species '{row['Species']}' retention time difference is {rt_diff} (within +/- {RETENTION_TIME_WINDOW}). Keeping row.")
            return True
        else:
            print(f"[DEBUG] Species '{row['Species']}' retention time difference is {rt_diff} (exceeds +/- {RETENTION_TIME_WINDOW}). Marking row for removal.")
            return False

# ------------------------------
# Process Each "Unmatched" File in the Input Directory
# ------------------------------

# Loop through all files in the input directory
for filename in os.listdir(INPUT_DIR):
    # Check if "unmatched" is in the filename and it is a CSV file
    if "unmatched" in filename and filename.lower().endswith('.csv'):
        nist_filepath = os.path.join(INPUT_DIR, filename)
        print(f"\n[INFO] Processing file: {nist_filepath}")

        # Load the NIST CSV file
        try:
            df_nist = pd.read_csv(nist_filepath)
            print(f"[DEBUG] Loaded NIST CSV from: {nist_filepath}")
            print(f"[DEBUG] NIST DataFrame shape: {df_nist.shape}")
        except Exception as e:
            print(f"[ERROR] Failed to load NIST CSV file {nist_filepath}: {e}")
            continue  # Skip to the next file

        # ------------------------------
        # Merge NIST with FAME on Species
        # ------------------------------
        try:
            merged = df_nist.merge(
                df_fame_unique[['Species', 'Retention_Time']],
                on='Species',
                how='left',
                suffixes=('', '_FAME')
            )
            print(f"[DEBUG] After merge, DataFrame shape: {merged.shape}")
        except Exception as e:
            print(f"[ERROR] Merge failed for {nist_filepath}: {e}")
            continue

        # ------------------------------
        # Check Retention Time for Each Row
        # ------------------------------
        merged['keep'] = merged.apply(check_retention_time, axis=1)

        # ------------------------------
        # Split the DataFrames Based on the Check
        # ------------------------------
        df_FAME_KEEP = merged[merged['keep']].copy()
        df_FAMEDONTKEEP = merged[~merged['keep']].copy()

        # Optionally drop extra columns from the outputs
        for df in [df_FAME_KEEP, df_FAMEDONTKEEP]:
            for col in ['Retention_Time_FAME', 'keep']:
                if col in df.columns:
                    df.drop(columns=[col], inplace=True)

        print(f"[DEBUG] df_FAME_KEEP shape: {df_FAME_KEEP.shape}")
        print(f"[DEBUG] df_FAMEDONTKEEP shape: {df_FAMEDONTKEEP.shape}")

        # ------------------------------
        # Save the Two DataFrames to CSV Files
        # ------------------------------
        # Use the base filename of the current file to generate output filenames
        base_filename = os.path.splitext(filename)[0]
        keep_filepath = os.path.join(OUTPUT_DIR, base_filename + KEEP_SUFFIX)
        dontkeep_filepath = os.path.join(OUTPUT_DIR, base_filename + DONT_KEEP_SUFFIX)

        try:
            df_FAME_KEEP.to_csv(keep_filepath, index=False)
            print(f"[DEBUG] Saved df_FAME_KEEP to: {keep_filepath}")
        except Exception as e:
            print(f"[ERROR] Failed to save df_FAME_KEEP for {nist_filepath}: {e}")

        try:
            df_FAMEDONTKEEP.to_csv(dontkeep_filepath, index=False)
            print(f"[DEBUG] Saved df_FAMEDONTKEEP to: {dontkeep_filepath}")
        except Exception as e:
            print(f"[ERROR] Failed to save df_FAMEDONTKEEP for {nist_filepath}: {e}")


[DEBUG] Successfully loaded FAME CSV from: Projects/FAME/analysis/OFF/off_possible/FAME_RT.csv
[DEBUG] FAME DataFrame shape (before deduplication): (10, 5)
[DEBUG] FAME DataFrame shape (after deduplication): (7, 5)

[INFO] Processing file: Projects/NIST/not_possible/unmatched_Blank.csv
[DEBUG] Loaded NIST CSV from: Projects/NIST/not_possible/unmatched_Blank.csv
[DEBUG] NIST DataFrame shape: (64, 41)
[DEBUG] After merge, DataFrame shape: (64, 42)
[DEBUG] Species '8:1' not found in FAME. Keeping row.
[DEBUG] Species '9:4' not found in FAME. Keeping row.
[DEBUG] Species '11:1' not found in FAME. Keeping row.
[DEBUG] Species '11:1' not found in FAME. Keeping row.
[DEBUG] Species '11:4' not found in FAME. Keeping row.
[DEBUG] Species '11:4' not found in FAME. Keeping row.
[DEBUG] Species '11:4' not found in FAME. Keeping row.
[DEBUG] Species '11:4' not found in FAME. Keeping row.
[DEBUG] Species '14:1' retention time difference is 0.2700833333333339 (within +/- 0.5). Keeping row.
[DEBUG] Sp

In [34]:
df_keep2 = pd.read_csv('Projects/NIST/isomer_filter/AFTER_NP/unmatched_NIST_n2_FAME_KEEP.csv')
df_dontkeep2 = pd.read_csv('Projects/NIST/isomer_filter/AFTER_NP/unmatched_NIST_n1_FAMEDONTKEEP.csv')

df_keep2

# look at 18:1 Species column
df_keep[df_keep['Species'].str.contains('18:1')]

,Lipid,Retention_Time,OzESI_Intensity,group_by_ion,group_by_lipid,Sample_ID,Transition,Sample,Parent_Ion,Product_Ion,...,Baseline_Drift,Shoulder_Count,Biology,Genotype,Cage,Mouse,Normalized_Peak_Area,n_value,n_position,OzOFF_Isomer
1138,FA(18:1)_<>_n-2,11.676433,2971.0,74,282,02032025_n1Plasma_AMP_ozoneonv2,437.4 -> 183.0,NIST_n1,437.4,183.0,...,5047.318612,0,NaN,NaN,NaN,NaN,0.002340,2,2.0,cis
1139,FA(18:1)_<>_n-2,11.755683,3714.0,74,282,02032025_n1Plasma_AMP_ozoneonv2,437.4 -> 183.0,NIST_n1,437.4,183.0,...,23697.160883,0,NaN,NaN,NaN,NaN,0.003938,2,2.0,cis
1140,FA(18:1)_<>_n-2,11.819083,2584.0,74,282,02032025_n1Plasma_AMP_ozoneonv2,437.4 -> 183.0,NIST_n1,437.4,183.0,...,3052.073877,0,NaN,NaN,NaN,NaN,0.002044,2,2.0,cis
1141,FA(18:1)_<>_n-2,11.945900,2850.0,74,282,02032025_n1Plasma_AMP_ozoneonv2,437.4 -> 183.0,NIST_n1,437.4,183.0,...,16775.054598,0,NaN,NaN,NaN,NaN,0.003131,2,2.0,cis
1142,FA(18:1)_<>_n-2,12.041017,3772.0,74,282,02032025_n1Plasma_AMP_ozoneonv2,437.4 -> 183.0,NIST_n1,437.4,183.0,...,2397.476341,0,NaN,NaN,NaN,NaN,0.002910,2,2.0,cis
1143,FA(18:1)_<>_n-4,12.058450,623.0,62,284,02032025_n1Plasma_AMP_ozoneonv2,409.3 -> 183.0,NIST_n1,409.3,183.0,...,1520.483771,1,NaN,NaN,NaN,NaN,0.001227,4,4.0,cis
1144,FA(18:1)_<>_n-5,11.773850,2737.0,56,285,02032025_n1Plasma_AMP_ozoneonv2,395.3 -> 183.0,NIST_n1,395.3,183.0,...,-349.946441,1,NaN,NaN,NaN,NaN,0.008128,5,5.0,cis
1145,FA(18:1)_<>_n-5,11.837267,2022.0,56,285,02032025_n1Plasma_AMP_ozoneonv2,395.3 -> 183.0,NIST_n1,395.3,183.0,...,5011.266336,0,NaN,NaN,NaN,NaN,0.001592,5,5.0,cis
1146,FA(18:1)_<>_n-5,11.916517,1507.0,56,285,02032025_n1Plasma_AMP_ozoneonv2,395.3 -> 183.0,NIST_n1,395.3,183.0,...,21324.921136,0,NaN,NaN,NaN,NaN,0.000541,5,5.0,cis
1147,FA(18:1)_<>_n-5,12.043333,1075.0,56,285,02032025_n1Plasma_AMP_ozoneonv2,395.3 -> 183.0,NIST_n1,395.3,183.0,...,2974.312753,0,NaN,NaN,NaN,NaN,0.000834,5,5.0,cis


In [28]:
df_dontkeep2
# Filte for Species 18:1


,Lipid,Retention_Time,OzESI_Intensity,group_by_ion,group_by_lipid,Sample_ID,Transition,Sample,Parent_Ion,Product_Ion,...,Genotype,Cage,Mouse,Normalized_Peak_Area,n_value,n_position,OzOFF_Isomer,Matched_Lipid_OFF,Intensity_OFF,Retention_Time_OFF
0,FA(14:1)_<>_n-4,12.061350,1729.0,38,103,02032025_n1Plasma_AMP_ozoneonv2,353.3 -> 183.0,NIST_n1,353.3,183.0,...,NaN,NaN,NaN,0.001814,4,4.0,trans,NaN,NaN,NaN
1,FA(14:1)_<>_n-7,11.698933,540.0,20,106,02032025_n1Plasma_AMP_ozoneonv2,311.3 -> 183.0,NIST_n1,311.3,183.0,...,NaN,NaN,NaN,0.000596,7,7.0,trans,NaN,NaN,NaN
2,FA(14:1)_<>_n-7,12.063500,1056.0,20,106,02032025_n1Plasma_AMP_ozoneonv2,311.3 -> 183.0,NIST_n1,311.3,183.0,...,NaN,NaN,NaN,0.001616,7,7.0,trans,NaN,NaN,NaN
3,FA(14:1)_<>_n-9,11.605117,1257.0,9,108,02032025_n1Plasma_AMP_ozoneonv2,283.2 -> 183.0,NIST_n1,283.2,183.0,...,NaN,NaN,NaN,0.000985,9,9.0,trans,NaN,NaN,NaN
4,FA(14:1)_<>_n-9,12.017267,795.0,9,108,02032025_n1Plasma_AMP_ozoneonv2,283.2 -> 183.0,NIST_n1,283.2,183.0,...,NaN,NaN,NaN,0.000796,9,9.0,trans,NaN,NaN,NaN
5,FA(14:1)_<>_n-10,11.605600,1313.0,5,98,02032025_n1Plasma_AMP_ozoneonv2,269.1 -> 183.0,NIST_n1,269.1,183.0,...,NaN,NaN,NaN,0.001002,10,10.0,trans,NaN,NaN,NaN
6,FA(16:1)_<>_n-5,9.682933,1806.0,44,150,02032025_n1Plasma_AMP_ozoneonv2,367.3 -> 183.0,NIST_n1,367.3,183.0,...,NaN,NaN,NaN,0.002012,5,5.0,trans,NaN,NaN,NaN
7,FA(16:1)_<>_n-6,9.651950,1408.0,38,151,02032025_n1Plasma_AMP_ozoneonv2,353.3 -> 183.0,NIST_n1,353.3,183.0,...,NaN,NaN,NaN,0.001586,6,6.0,trans,NaN,NaN,NaN
8,FA(16:1)_<>_n-8,9.621683,4345.0,26,153,02032025_n1Plasma_AMP_ozoneonv2,325.2 -> 183.0,NIST_n1,325.2,183.0,...,NaN,NaN,NaN,0.003100,8,8.0,trans,NaN,NaN,NaN
9,FA(16:1)_<>_n-9,9.669950,3914.0,20,154,02032025_n1Plasma_AMP_ozoneonv2,311.3 -> 183.0,NIST_n1,311.3,183.0,...,NaN,NaN,NaN,0.004306,9,9.0,trans,NaN,NaN,NaN


# Bar plot of each species

In [ ]:
df_keep2 = pd.read_csv('Projects/NIST/isomer_filter/AFTER_NP/unmatched_NIST_n2_FAME_KEEP.csv')

In [33]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------
# User-Defined Variables
# ------------------------------

# Directory where the FAME_KEEP CSV files are stored
INPUT_DIR = 'Projects/NIST/isomer_filter/AFTER_NP/'

# ------------------------------
# Process Each FAME_KEEP CSV File
# ------------------------------

# List through all files in the input directory that end with '_FAME_KEEP.csv'
for filename in os.listdir(INPUT_DIR):
    if filename.endswith('_FAME_KEEP.csv'):
        file_path = os.path.join(INPUT_DIR, filename)
        print(f"[INFO] Processing file: {file_path}")
        
        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"[ERROR] Could not read {file_path}: {e}")
            continue

        # Create a new subdirectory for the plots
        # Example: if filename is "unmatched_NIST_n2_FAME_KEEP.csv", the subdir will be "plot_unmatched_NIST_n2_FAME_KEEP"
        base_filename = os.path.splitext(filename)[0]
        plot_subdir = os.path.join(INPUT_DIR, f"plot_{base_filename}")
        os.makedirs(plot_subdir, exist_ok=True)
        print(f"[INFO] Created/Found plot directory: {plot_subdir}")
        
        # Check that necessary columns exist
        required_columns = ['Species', 'Lipid', 'OzESI_Intensity']
        if not all(col in df.columns for col in required_columns):
            print(f"[WARNING] The file {filename} does not contain the required columns {required_columns}. Skipping.")
            continue
        
        # Get unique species from the dataframe
        unique_species = df['Species'].unique()
        print(f"[DEBUG] Found {len(unique_species)} unique species in {filename}.")
        
        # For each unique species, create a bar plot
        for species in unique_species:
            # Filter the dataframe for the current species
            df_species = df[df['Species'] == species]
            
            # Create the bar plot:
            #   x-axis: Lipid values
            #   y-axis: OzESI_Intensity values
            plt.figure(figsize=(10, 6))
            plt.bar(df_species['Lipid'], df_species['OzESI_Intensity'], color='skyblue')
            plt.xlabel('Lipid', fontsize=12)
            plt.ylabel('OzESI_Intensity', fontsize=12)
            plt.title(f"Species: {species}", fontsize=14)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()

            # Sanitize species string for use in a filename (replace problematic characters)
            safe_species = species.replace(":", "_").replace(" ", "_").replace("/", "_")
            plot_filename = f"{safe_species}.png"
            plot_filepath = os.path.join(plot_subdir, plot_filename)
            
            try:
                plt.savefig(plot_filepath)
                print(f"[INFO] Saved plot for species '{species}' to {plot_filepath}")
            except Exception as e:
                print(f"[ERROR] Could not save plot for species '{species}': {e}")
            finally:
                plt.close()  # Close the figure to free up memory


[INFO] Processing file: Projects/NIST/isomer_filter/AFTER_NP/unmatched_NIST_n3_FAME_KEEP.csv
[INFO] Created/Found plot directory: Projects/NIST/isomer_filter/AFTER_NP/plot_unmatched_NIST_n3_FAME_KEEP
[DEBUG] Found 35 unique species in unmatched_NIST_n3_FAME_KEEP.csv.
[INFO] Saved plot for species '6:1' to Projects/NIST/isomer_filter/AFTER_NP/plot_unmatched_NIST_n3_FAME_KEEP/6_1.png
[INFO] Saved plot for species '7:1' to Projects/NIST/isomer_filter/AFTER_NP/plot_unmatched_NIST_n3_FAME_KEEP/7_1.png
[INFO] Saved plot for species '8:1' to Projects/NIST/isomer_filter/AFTER_NP/plot_unmatched_NIST_n3_FAME_KEEP/8_1.png
[INFO] Saved plot for species '9:1' to Projects/NIST/isomer_filter/AFTER_NP/plot_unmatched_NIST_n3_FAME_KEEP/9_1.png
[INFO] Saved plot for species '9:2' to Projects/NIST/isomer_filter/AFTER_NP/plot_unmatched_NIST_n3_FAME_KEEP/9_2.png
[INFO] Saved plot for species '10:1' to Projects/NIST/isomer_filter/AFTER_NP/plot_unmatched_NIST_n3_FAME_KEEP/10_1.png
[INFO] Saved plot for specie